# WOTB Tank Recommendation System Using Content-Based Filtering

In this project, we develop a content-based filtering system to recommend tanks in the World of Tanks Blitz (WOTB) mobile game, based on tank characteristics.
Tank data is retrieved from the Wargaming API, then organised as a dataframe.
The selected numerical and categorical features are then preprocessed and used to calculate cosine similarity between tanks.
Based on these similarity scores, the system recommends tanks that are most similar to a tank selected by the user.

Content-based filtering was chosen because the available data provides detailed information about the features and characteristics of each tank, but does not include user ratings, preferences, or historical interactions.
Other approaches such as collaborative filtering generally require user-item interaction data to learn recommendations from the behaviour of multiple users.
Since this information is not available, content-based filtering is more suitable for this dataset.

In [62]:
# import libraries required
import requests
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# API key can be obtained from Wargaming's developer room website
import os
from dotenv import load_dotenv

load_dotenv()

APPLICATION_ID = os.getenv("APPLICATION_ID")

To obtain data of tanks, we query Wargaming's World of Tanks Blitz API through the `encyclopedia/vehicles` endpoint.
For more details on the API, refer to the [Wargaming API documentation](https://developers.wargaming.net/reference/all/wotb/encyclopedia/vehicles/?r_realm=asia).

In [64]:
url = "https://api.wotblitz.asia/wotb/encyclopedia/vehicles/"

params = {
    "application_id": APPLICATION_ID,
    "language": "en",
}

response = requests.get(url, params=params, timeout=30)
response.raise_for_status()

In [65]:
data = response.json()

In [66]:
tanks = data['data']

In [67]:
# see the first tank
first_tank = list(tanks.values())[0]
first_tank

{'suspensions': [12290],
 'description': None,
 'engines': [74757],
 'prices_xp': {'10753': 236250},
 'next_tanks': None,
 'modules_tree': {'12290': {'name': 'IS-4',
   'next_modules': None,
   'next_tanks': None,
   'is_default': True,
   'price_xp': 0,
   'price_credit': 0,
   'module_id': 12290,
   'type': 'vehicleChassis'},
  '10243': {'name': 'IS-4M',
   'next_modules': None,
   'next_tanks': None,
   'is_default': True,
   'price_xp': 0,
   'price_credit': 0,
   'module_id': 10243,
   'type': 'vehicleTurret'},
  '268036': {'name': '122 mm M62 IS4',
   'next_modules': None,
   'next_tanks': None,
   'is_default': True,
   'price_xp': 0,
   'price_credit': 0,
   'module_id': 268036,
   'type': 'vehicleGun'},
  '74757': {'name': 'V12 IS4',
   'next_modules': None,
   'next_tanks': None,
   'is_default': True,
   'price_xp': 0,
   'price_credit': 0,
   'module_id': 74757,
   'type': 'vehicleEngine'}},
 'nation': 'ussr',
 'is_premium': False,
 'images': {'preview': 'https://glossary-a

The API response is in the form of a nested JSON, so we extract and organise the data into a pandas dataframe.

In [68]:
def extract_tank_features(tanks):
    rows = []

    for tank_id, tank in tanks.items():

        profile = tank.get("default_profile", {})
        gun = profile.get("gun", {})
        turret = profile.get("turret", {})
        engine = profile.get("engine", {})
        suspension = profile.get("suspension", {})
        armor = profile.get("armor", {})
        hull_armor = armor.get("hull", {})
        turret_armor = armor.get("turret", {})
        shells = profile.get("shells", [])

        # tank information
        row = {
            "name": tank.get("name"), # Vehicle name
            "nation": tank.get("nation"), # Nation
            "type": tank.get("type"), # Vehicle type
            "tier": tank.get("tier"), # Tier
            "is_premium": tank.get("is_premium"), # Indicates if the vehicle is Premium vehicle

            "hp": profile.get("hp", 0), # Hit points
            "speed_forward": profile.get("speed_forward", 0), # Top speed (km/h)
            "speed_backward": profile.get("speed_backward", 0), # Top reverse speed (km/h)
            "weight": profile.get("weight", 0), # Weight (kg)

            "hull_armor_front": hull_armor.get("front", 0), # Front hull armor (mm)
            "hull_armor_rear": hull_armor.get("rear", 0), # Rear hull armor (mm)
            "hull_armor_side": hull_armor.get("sides", 0), # Side hull armor (mm)
            "turret_armor_front": turret_armor.get("front", 0), # Front turret armor (mm)
            "turret_armor_rear": turret_armor.get("rear", 0), # Rear turret armor (mm)
            "turret_armor_side": turret_armor.get("sides", 0), # Side turret armor (mm)

            "engine_power": engine.get("power", 0),

            "aim_time": gun.get("aim_time", 0), # Aiming time (s)
            "caliber": gun.get("caliber", 0), # Caliber (mm)
            "clip_capacity": gun.get("clip_capacity", 1), # Number of shells in the ammo
            "clip_reload_time": gun.get("clip_reload_time", 0), # Reload time
            "gun_depression": gun.get("move_down_arc", 0), # Depression angle (deg)
            "gun_elevation": gun.get("move_up_arc", 0), # Elevation angle (deg)
            "dispersion": gun.get("dispersion", 0), # Dispersion at 100 m (m)
            "fire_rate": gun.get("fire_rate", 0), # Rate of fire (rounds/min)
            "reload_time": gun.get("reload_time", 0), # Reload time (s)

            "hull_traverse": suspension.get("traverse_speed", 0), # Traverse speed (deg/s)

            "turret_traverse": turret.get("traverse_speed", 0), # Traverse speed (deg/s)
            "view_range": turret.get("view_range", 0), # View range (m)

            # derived features
            "power_to_weight": engine.get("power", 0) / profile.get("weight", 1),
            "avg_damage": np.mean([shell.get("damage", 0) for shell in shells]),
            "avg_penetration": np.mean([shell.get("penetration", 0) for shell in shells]),
        }

        rows.append(row)

    return pd.DataFrame(rows)

In [69]:
def add_features(df):

    df = df.copy()

    df["dpm"] = df["avg_damage"] * df["fire_rate"]
    df["hull_armor_avg"] = (df["hull_armor_front"] + df["hull_armor_side"] + df["hull_armor_rear"]) / 3
    df["turret_armor_avg"] = (df["turret_armor_front"] + df["turret_armor_side"] + df["turret_armor_rear"]) / 3
    df["burst_damage"] = df["avg_damage"] * df["clip_capacity"]
    df["gun_handling"] = (1 / (df["aim_time"] * df["dispersion"]).replace(0, np.nan))

    return df.replace([np.inf, -np.inf], np.nan).fillna(0)

In [70]:
tank_df = extract_tank_features(tanks)
tank_df = add_features(tank_df)
tank_df.head()

,name,nation,type,tier,is_premium,hp,speed_forward,speed_backward,weight,hull_armor_front,...,turret_traverse,view_range,power_to_weight,avg_damage,avg_penetration,dpm,hull_armor_avg,turret_armor_avg,burst_damage,gun_handling
0,IS-4,ussr,heavyTank,10,False,2650,38,14,60671,160,...,14,250,0.011538,426.666667,222.000000,2325.333333,133.333333,196.666667,426.666667,0.459897
1,T-34,ussr,mediumTank,5,False,620,48,18,28432,50,...,40,200,0.014069,130.000000,100.000000,939.900000,48.333333,55.000000,130.000000,0.358192
2,Waffenträger auf Pz. IV,germany,AT-SPG,9,False,1600,40,12,26850,80,...,18,250,0.013408,483.333333,207.333333,2634.166667,43.333333,3.333333,483.333333,0.629327
3,Super Hellcat,usa,AT-SPG,7,True,1100,65,20,20378,25,...,20,240,0.022573,255.000000,156.666667,2636.700000,25.000000,52.000000,255.000000,0.687569
4,T49,usa,lightTank,8,False,1100,72,24,24011,25,...,40,250,0.020824,231.666667,156.000000,2043.300000,23.000000,25.000000,231.666667,0.496524


The selected features contain categorical and numerical variables.
Categorical features are one-hot encoded, while numerical features are standardised using `StandardScaler`.

In [71]:
# features
categorical_features = [
    "nation",
    "type"
]

numerical_features = [
    "tier",
    "hp",

    # gun
    "avg_damage",
    "avg_penetration",
    "caliber",
    "dpm",
    "reload_time",
    "gun_elevation",
    "gun_depression",
    "gun_handling",
    "clip_capacity",
    "clip_reload_time",
    "burst_damage",

    # armour
    "hull_armor_avg",
    "turret_armor_avg",

    # mobility
    "speed_forward",
    "speed_backward",
    "power_to_weight",
    "hull_traverse",
    "turret_traverse",

    "view_range",
]

# one-hot encode the categorical values,
# standardise the numerical values
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

X = preprocessor.fit_transform(tank_df)

Cosine similarity is used to measure how similar each pair of tanks is based on their processed features.
The resulting matrix contains a similarity score for every pair of tanks.

In [72]:
# calculate similarity between tanks
similarity_matrix = cosine_similarity(X)
print(similarity_matrix.shape)

(529, 529)


Finally, we sort tanks based on the cosine similarity score, and return the top k most similar tanks to a given tank.

In [73]:
def recommend_tanks(tank_name,
                    tank_df,
                    similarity_matrix,
                    k=10,
                    exclude_premium=False):

    # find tank by exact name first
    matches = tank_df[tank_df["name"].str.lower() == tank_name.lower()]

    # if no exact match, allow partial matching
    if matches.empty:
        matches = tank_df[tank_df["name"].str.contains(tank_name, case=False, na=False)]

    if matches.empty:
        print(f"Tank {tank_name} was not found")
        return

    matched_name = matches.iloc[0]["name"]

    tank_index = matches.index[0]

    similarities = similarity_matrix[tank_index]

    # sort in descending order of similarity
    similar_indices = np.argsort(similarities)[::-1]

    # remove the tank itself
    similar_indices = [i for i in similar_indices if i != tank_index]

    # exclude premium tanks before taking top k
    if exclude_premium:
        similar_indices = [
            i for i in similar_indices
            if tank_df.iloc[i]["is_premium"] == 0
        ]

    # take top k
    similar_indices = similar_indices[:k]
    recommendations = tank_df.iloc[similar_indices].copy()
    recommendations["similarity"] = [similarities[i] for i in similar_indices]

    print(f"Because you liked {matched_name}, you might also enjoy:")
    for index, (_, tank) in enumerate(recommendations.iterrows(), start=1):
        print(f"{index}. {tank['name']} (Tier {tank['tier']} {tank['type']}, similarity {tank['similarity']:.3f})")

In [74]:
recommend_tanks(
    "smasher",
    tank_df,
    similarity_matrix,
    k=10,
    exclude_premium=True,
)

Because you liked Smasher, you might also enjoy:
1. FV215b (183) (Tier 10 AT-SPG, similarity 0.894)
2. SU-152 (Tier 7 AT-SPG, similarity 0.846)
3. ISU-152 (Tier 8 AT-SPG, similarity 0.835)
4. Jagdpanzer E 100 (Tier 10 AT-SPG, similarity 0.811)
5. T110E4 (Tier 10 AT-SPG, similarity 0.811)
6. WZ-110 (Tier 8 heavyTank, similarity 0.797)
7. VK 72.01 (K) (Tier 10 heavyTank, similarity 0.784)
8. Object 704 (Tier 9 AT-SPG, similarity 0.778)
9. IS-3 (Tier 8 heavyTank, similarity 0.778)
10. IS-8 (Tier 9 heavyTank, similarity 0.769)


Performing a sanity check on the top 10 recommendations, I believe the recommendations make sense.

Possible future work can include:
- Integrate player data from the Wargaming API to incorporate individual play history and preferences.
- Use 30-day or 90-day statistics such as win rate, average damage, and battles played to personalise recommendations.
- Evaluate the performance of system using real player data and feedback.